In [6]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import timm
from dataclasses import dataclass
from tqdm import tqdm
import os

/Users/erroldmello/Documents/college/pawlensai/roboflowvenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
@dataclass
class Config:
    data_dir: str = "dataset"
    model_name: str = "vit_base_patch16_224"
    batch_size: int = 32
    epochs: int = 15
    lr: float = 2e-4
    image_size: int = 224
    num_classes: int = 6
    weight_decay: float = 1e-4
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

cfg = Config()

In [8]:
# ---------------- TRANSFORMS ----------------
train_tfms = transforms.Compose([
    transforms.Resize((cfg.image_size, cfg.image_size)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

val_tfms = transforms.Compose([
    transforms.Resize((cfg.image_size, cfg.image_size)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

In [9]:
# ---------------- DATA ----------------
train_ds = datasets.ImageFolder(os.path.join(cfg.data_dir, "train"), transform=train_tfms)
val_ds   = datasets.ImageFolder(os.path.join(cfg.data_dir, "valid"), transform=val_tfms)

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=cfg.batch_size)

In [10]:
# ---------------- MODEL ----------------
model = timm.create_model(
    cfg.model_name,
    pretrained=True,
    num_classes=cfg.num_classes
).to(cfg.device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg.lr,
    weight_decay=cfg.weight_decay
)

In [6]:
# ---------------- TRAIN FUNCTION ----------------
def train_model():
    best_acc = 0

    for epoch in range(cfg.epochs):
        model.train()
        epoch_loss = 0

        for imgs, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            imgs, labels = imgs.to(cfg.device), labels.to(cfg.device)

            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()

        # Validation
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(cfg.device), labels.to(cfg.device)
                preds = model(imgs).argmax(dim=1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)

        acc = correct / total
        print(f"Epoch {epoch+1} | Loss: {epoch_loss:.4f} | Val Acc: {acc:.4f}")

        if acc > best_acc:
            best_acc = acc
            torch.save(model.state_dict(), "best_vit_model.pth")

train_model()

Epoch 1: 100%|██████████| 93/93 [09:50<00:00,  6.35s/it]


Epoch 1 | Loss: 139.8339 | Val Acc: 0.5876


Epoch 2: 100%|██████████| 93/93 [09:37<00:00,  6.21s/it]


Epoch 2 | Loss: 86.9583 | Val Acc: 0.6811


Epoch 3: 100%|██████████| 93/93 [09:53<00:00,  6.38s/it]


...

Epoch 13 | Loss: 13.3162 | Val Acc: 0.8785


Epoch 14: 100%|██████████| 93/93 [09:25<00:00,  6.08s/it]


Epoch 14 | Loss: 16.8050 | Val Acc: 0.9042


Epoch 15: 100%|██████████| 93/93 [09:36<00:00,  6.20s/it]


Epoch 15 | Loss: 14.0420 | Val Acc: 0.8890


In [11]:
# ---------------- CLASS NAMES ----------------
CLASS_NAMES = [
    "demodicosis",
    "dermatitis",
    "fungal_infections",
    "healthy",
    "hypersensitivity",
    "ringworm"
]

In [12]:
# ---------------- TEST DATASET ----------------
test_ds = datasets.ImageFolder(
    os.path.join(cfg.data_dir, "test"),
    transform=val_tfms   # use validation transforms (NO augmentation)
)

test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False)

In [13]:
# ---------------- TEST EVALUATION ----------------
# Load saved best weights
checkpoint_path = "best_vit_model.pth"
if os.path.exists(checkpoint_path):
    state = torch.load(checkpoint_path, map_location=cfg.device)
    try:
        model.load_state_dict(state)
    except Exception:
        model = state
    model.to(cfg.device)
    print(f"Loaded checkpoint: {checkpoint_path}")
else:
    print(f"Warning: checkpoint not found at {checkpoint_path}. Using current model weights.")

model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(cfg.device), labels.to(cfg.device)
        outputs = model(images)
        preds = outputs.argmax(dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

test_acc = correct / total if total > 0 else 0.0
print(f"\nTest Accuracy: {test_acc:.4f}")

Loaded checkpoint: best_vit_model.pth

Test Accuracy: 0.9118


In [14]:
# ---------------- PER-CLASS ACCURACY & CONFUSION MATRIX ----------------
import numpy as np

# collect predictions and labels over the test set
all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(cfg.device)
        outputs = model(images)
        preds = outputs.argmax(dim=1).cpu()

        all_preds.append(preds)
        all_labels.append(labels.cpu())

if len(all_preds) == 0:
    print("No predictions collected from test_loader.")
else:
    all_preds = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    num_classes = len(CLASS_NAMES)

    # Build confusion matrix
    cm = np.zeros((num_classes, num_classes), dtype=int)
    for t, p in zip(all_labels, all_preds):
        cm[int(t), int(p)] += 1

    # Per-class accuracy
    per_class_acc = []
    for i in range(num_classes):
        total = cm[i].sum()
        correct = cm[i, i]
        acc = float(correct) / total if total > 0 else 0.0
        per_class_acc.append(acc)

    print("\nPer-class accuracy:")
    for i, acc in enumerate(per_class_acc):
        name = CLASS_NAMES[i]
        print(f"  {name}: {acc:.4f} ({cm[i,i]}/{cm[i].sum()})")

    print("\nConfusion matrix (rows=true, cols=pred):")
    print(cm)

    # Classification report (requires sklearn)
    try:
        from sklearn.metrics import classification_report

        print("\nClassification report:")
        print(classification_report(
            all_labels,
            all_preds,
            target_names=CLASS_NAMES,
            zero_division=0
        ))
    except Exception as e:
        print("\nCould not generate classification report:", e)


Per-class accuracy:
  demodicosis: 0.8939 (59/66)
  dermatitis: 0.9057 (48/53)
  fungal_infections: 0.8696 (60/69)
  healthy: 0.6429 (18/28)
  hypersensitivity: 0.9900 (99/100)
  ringworm: 0.9478 (109/115)

Confusion matrix (rows=true, cols=pred):
[[ 59   2   1   0   1   3]
 [  1  48   2   0   2   0]
 [  0   9  60   0   0   0]
 [  1   5   4  18   0   0]
 [  0   1   0   0  99   0]
 [  1   2   1   0   2 109]]

Classification report:
                   precision    recall  f1-score   support

      demodicosis       0.95      0.89      0.92        66
       dermatitis       0.72      0.91      0.80        53
fungal_infections       0.88      0.87      0.88        69
          healthy       1.00      0.64      0.78        28
 hypersensitivity       0.95      0.99      0.97       100
         ringworm       0.97      0.95      0.96       115

         accuracy                           0.91       431
        macro avg       0.91      0.87      0.89       431
     weighted avg       0.92   